In [0]:
%pip install confluent-kafka
%pip install confluent-kafka[avro,schemaregistry]
%restart_python

In [0]:
"""
Productor Kafka simple de eventos de pago aleatorios.

Ejecutar en la primera celda:
    %pip install confluent-kafka
    %pip install confluent-kafka[avro,schemaregistry]
    %restart_python

El productor genera eventos indefinidamente. Detener con Ctrl+C.
"""

import random  # Permite elegir préstamos, cuotas y valores al azar.
import time  # Permite esperar entre eventos.
import uuid  # Genera un identificador único para cada evento.
from datetime import datetime, timezone  # Obtiene la fecha y hora actual en UTC.
from pathlib import Path  # Permite ubicar el archivo del esquema Avro.

from confluent_kafka import Producer  # Cliente que publica mensajes en Kafka.
from confluent_kafka.schema_registry import SchemaRegistryClient  # Conecta con Schema Registry.
from confluent_kafka.schema_registry.avro import AvroSerializer  # Convierte eventos a Avro.
from confluent_kafka.serialization import MessageField, SerializationContext  # Indican que el Avro es un valor.

## Parámetros de Entrada

In [0]:
# ============================================================
# CONFIGURACIÓN: modificar estos valores antes de ejecutar
# ============================================================
KAFKA_BOOTSTRAP_SERVERS = "kafka-utec-pe-01-utec-data.b.aivencloud.com:28780"  # Servidores Kafka.
KAFKA_SECURITY_PROTOCOL = "SASL_SSL"  # Protocolo seguro de conexión.
KAFKA_SASL_MECHANISMS = "SCRAM-SHA-256"  # Mecanismo de autenticación de Aiven.
KAFKA_SASL_USERNAME = "avnadmin"  # Usuario o API key de Kafka.
KAFKA_SASL_PASSWORD = dbutils.secrets.get(catalog="dev_bronze", schema="kafka_streams", key="KAFKA_SASL_PASSWORD") # Contraseña o API secret de Kafka.

# Certificado CA de Aiven en formato PEM.
# Pegar el contenido completo, incluyendo BEGIN CERTIFICATE y END CERTIFICATE.
CA_CERTIFICATE = dbutils.fs.head("/Volumes/dev_bronze/kafka_streams/vol_streams/pem/ca.pem", 1048576)

SCHEMA_REGISTRY_URL = "https://kafka-utec-pe-01-utec-data.b.aivencloud.com:28772"  # URL del Schema Registry.
SCHEMA_REGISTRY_USER= "avnadmin"  # Usuario del Registry.
SCHEMA_REGISTRY_PWD = dbutils.secrets.get(catalog="dev_bronze", schema="kafka_streams", key="SCHEMA_REGISTRY_PASSWORD") # Contraseña o API secret de Kafka.  # Secreto del Registry.

SCHEMA_AVRO = """
{
  "type": "record",
  "name": "PagoEvento",
  "namespace": "banco.pagos",
  "fields": [
    {"name": "evento_id", "type": "string"},
    {"name": "cuota_id", "type": "long"},
    {"name": "prestamo_id", "type": "long"},
    {"name": "fecha_pago", "type": {"type": "int", "logicalType": "date"}},
    {"name": "monto_pagado", "type": "double"},
    {"name": "metodo_pago_codigo", "type": "string"},
    {"name": "canal_codigo", "type": "string"},
    {"name": "timestamp_evento", "type": {"type": "long", "logicalType": "timestamp-millis"}}
  ]
}
"""

KAFKA_TOPIC = "banco.pagos"  # Topic donde se publican los eventos.
EVENT_INTERVAL_SECONDS = 3.0  # Segundos de espera entre eventos.

# Se elige primero un préstamo y después una cuota de su propia lista.
# Así nunca se envía una combinación inválida de prestamo_id y cuota_id.
# No cambiar estos IDs sin actualizar también sql/dml.sql.
PRESTAMOS = {
    1001: [100101, 100102, 100103, 100104],
    1002: [100201, 100202, 100203, 100204],
    1003: [100301, 100302],
    1004: [100401, 100402, 100403, 100404],
    1005: [100501, 100502, 100503, 100504, 100505, 100506],
    1006: [100601],
    1007: [100701],
    1008: [100801],
    1009: [100901],
    1010: [101001],
    1011: [101101],
    1012: [101201],
    1013: [101301],
    1014: [101401],
    1015: [101501],
    1016: [101601],
    1017: [101701],
    1018: [101801],
    1019: [101901],
    1020: [102001],
    1021: [102101],
    1022: [102201],
    1023: [102301],
    1024: [102401],
    1025: [102501],
}

METODOS_PAGO = ["TRANS", "VENT", "DEBI"]  # Códigos de cat_metodo_pago.csv.
CANALES = ["APP", "WEB", "VENT"]  # Códigos de cat_canal.csv.

In [0]:
def main():
    """Conecta con Kafka y publica eventos de pago aleatorios continuamente."""

    # Se lee el esquema Avro que define los campos del evento pago_evento.
    schema = SCHEMA_AVRO

    # Se crea el productor con los datos de conexión definidos arriba.
    producer = Producer(
        {
            "bootstrap.servers": KAFKA_BOOTSTRAP_SERVERS,
            "security.protocol": KAFKA_SECURITY_PROTOCOL,
            "sasl.mechanisms": KAFKA_SASL_MECHANISMS,
            "sasl.username": KAFKA_SASL_USERNAME,
            "sasl.password": KAFKA_SASL_PASSWORD,
            "ssl.ca.pem": CA_CERTIFICATE,
            "client.id": "banco-pagos-random-producer",
        }
    )

    # Schema Registry almacena y valida el esquema usado para serializar Avro.
    schema_registry = SchemaRegistryClient(
        {
            "url": SCHEMA_REGISTRY_URL,
            "basic.auth.user.info": (
                f"{SCHEMA_REGISTRY_USER}:{SCHEMA_REGISTRY_PWD}"
            ),
        }
    )
    serializer = AvroSerializer(schema_registry, schema)

    print(f"Productor iniciado en {KAFKA_TOPIC}. Presiona Ctrl+C para detenerlo.")
    contador = 0

    try:
        while True:
            # Se selecciona un préstamo existente y una de sus cuotas existentes.
            evento_id = f"EVT-RANDOM-{uuid.uuid4().hex[:12].upper()}"
            prestamo_id = random.choice(list(PRESTAMOS))
            cuota_id = random.choice(PRESTAMOS[prestamo_id])
            ahora = datetime.now(timezone.utc)

            # Se construye el mensaje con la estructura definida en pago_evento.avsc.
            evento = {
                "evento_id": evento_id,
                "cuota_id": cuota_id,
                "prestamo_id": prestamo_id,
                "fecha_pago": ahora.date(),
                # Puede ser diferente de cuota.monto_programado.
                "monto_pagado": round(random.uniform(50.00, 5000.00), 2),
                "metodo_pago_codigo": random.choice(METODOS_PAGO),
                "canal_codigo": random.choice(CANALES),
                "timestamp_evento": int(ahora.timestamp() * 1000),
            }

            # Se envía el evento a Kafka.
            # La key usa prestamo_id para agrupar el mismo préstamo en una partición.
            producer.produce(
                topic=KAFKA_TOPIC,
                key=str(prestamo_id),
                value=serializer(
                    evento,
                    SerializationContext(KAFKA_TOPIC, MessageField.VALUE),
                ),
            )

            # Atiende las confirmaciones de entrega pendientes del productor.
            producer.poll(0)
            contador += 1
            print(
                f"Evento #{contador}: evento_id={evento_id}, prestamo={prestamo_id}, "
                f"cuota={cuota_id}, monto={evento['monto_pagado']:.2f}"
            )
            # Espera antes de crear el siguiente evento.
            time.sleep(EVENT_INTERVAL_SECONDS)
    except KeyboardInterrupt:
        # Ctrl+C permite detener el ciclo infinito de forma controlada.
        print(f"\nProductor detenido. Eventos generados: {contador}")
    finally:
        # Espera hasta 10 segundos para entregar los mensajes pendientes.
        producer.flush(10)

main()